In [ ]:
from phase_II.nifty_re_playground.strain_tools import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib tk

## Calibrated strain time series containing GW150914

In [8]:
nrt_strain_values = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_strain_values.txt") * 1e19
nrt_time_values = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_time_values.txt")

# -- Set a GPS time:
t0 = 1126259462.4    # -- GW150914
#-- Choose detector as H1, L1, or V1

strain = unpickle_me_this("/Users/iason/PycharmProjects/STRAIN/phase_I/partial_successful_reconstruct_and_where_is_the_signal/store/GW150914_strain.pickle", absolute_path=True)
data_ar = 1e19 * strain.value

zero_time = 1.1262594e9 + 60 + 2.422 + 0.00109 # I got this zero time by looking at the caption of the figure produced by strain.plot().
time_nr_template = convert_gps_to_seconds(nrt_time_values, t0=zero_time, )
time_strain_data = convert_gps_to_seconds(strain.times, t0=zero_time)

# plt.figure(figsize=(8,4))
# plt.plot(time_strain_data, data_ar, "-", color="black", lw=1)
# thesis_plot(save_fig=False)

In [ ]:
_, axs = plt.subplots(nrows=2, ncols=1, sharex=False, figsize=(8,2+2))
axs[0].plot(time_nr_template, nrt_strain_values, color="black")
axs[1].plot(time_strain_data, data_ar, color="black")

axs[0].set_xlim(-0.2, 0.036)
axs[0].set_ylim(-0.015, 0.015)

axs[0].set_ylabel(r"$h(t)$ $\:\mathrm{[10^{-19}]}$")
axs[1].set_ylabel(r"$d(t)$ $\:\mathrm{[10^{-19}]}$")
axs[1].set_xlabel(r"Time $t$ $\mathrm{[s]}$")

save_figure(False)

## Welch averaged PSD

In [ ]:
# events = [
#     {'desired_duration': 32, 'unpack': True, 'gps_center': 1126259462.4,
#      'absolute_path':
#          '/Users/iason/PycharmProjects/STRAIN/data/data_pickle_or_hdf5/gwpy_objects_II/H1_GW150914.hdf5'
#      },
#     {
#     'desired_duration': 32, 'unpack': True, 'gps_center': 1242459857.4,
#      'absolute_path':
#          '/Users/iason/PycharmProjects/STRAIN/data/data_pickle_or_hdf5/gwpy_objects_II/L1_GW190521_074359'
#     },
# ]
#
# time_event_1, strain_event_1 = _get(**events[0])
# time_event_2, strain_event_2 = _get_strain_data(**events[1])

In [ ]:
GW150914 = get_time_and_strain_from_disc()
GW190521_074359 = get_time_and_strain_from_disc("GW190521_074359", detector="L1")

time_event_1, strain_event_1 = GW150914.time, GW150914.strain
time_event_2, strain_event_2 = GW190521_074359.time, GW190521_074359.strain

In [ ]:
import jax.numpy as jnp

from scipy.signal.windows import tukey, hann

tukey_taper = lambda d: tukey(M=len(d), alpha=0.1, sym=True)
hann_taper = lambda d: hann(M=len(d), sym=True)


# Build skeleton
_, axs = plt.subplots(nrows=4, ncols=1, sharex=True, sharey=True, figsize=(8,4*4))

axs = np.array(axs).reshape(2,2)

for ax in axs:
    ax[0].loglog()

axs[0][0].set_ylabel(r"Noise power")
axs[0][1].set_ylabel(r"Noise power")
axs[1][0].set_ylabel(r"Noise power")
axs[1][0].set_ylabel(r"Noise power")
axs[1][1].set_xlabel(r"Frequency $f$ $\mathrm{[Hz]}$")


# Fill: upper left plot
x_ul, y_ul, _ = calculate_welch_average(x=time_event_1, y=strain_event_1, L=2, final_average_call=jnp.mean, tapering_function=tukey_taper)

axs[0][0].plot(x_ul, y_ul, color="black")
axs[0][0].annotate(
    "",
    xy=(25.8, 2.39e-05),
    xytext=(7.47, 0.464),
    arrowprops=dict(
        arrowstyle="->",
        connectionstyle="arc3,rad=0.1"
    )
)
axs[0][0].text(2.1, 1.2e-3, "Seismic wall")
axs[0][0].text(10, 2.59e-7, "Thermal noise")
axs[0][0].annotate(
    "",
    xy=(125.53, 9.838e-7),
    xytext=(28.6, 1.79e-5),
    arrowprops=dict(
        arrowstyle="->",
        connectionstyle="arc3,rad=0.2"
    )
)
axs[0][0].text(300, 5e-8, "Photon shot\n noise")
axs[0][0].annotate(
    "",
    xy=(1394, 8e-6),
    xytext=(162, 8e-7),
    arrowprops=dict(
        arrowstyle="->",
        connectionstyle="arc3,rad=0.05"
    )
)
axs[0][0].annotate(
    "Power grid",
    xy=(62.0173, 0.17634),
    xytext=(47.9154, 1296.76),
    arrowprops=dict(
        arrowstyle="->",
        connectionstyle="arc3,rad=0."
    )
)
axs[0][0].annotate(
    "Violin modes",
    xy=(323, 0.145),
    xytext=(233, 1177),
    arrowprops=dict(
        arrowstyle="->",
        connectionstyle="arc3,rad=0."
    )
)


# Fill: upper right plot
x_ur, y_ur, _ = calculate_welch_average(x=time_event_1, y=strain_event_1, L=2, final_average_call=jnp.mean, tapering_function=hann_taper)

axs[0][1].plot(x_ul, y_ul, color="black", alpha=1)
axs[0][1].plot(x_ur, y_ur, label="GW150914, Hann-windowed", color=blue, alpha=0.5)
axs[0][1].legend(loc="lower left")


# Fill: Lower left plot
x_ll, y_ll, _ = calculate_welch_average(x=time_event_2, y=strain_event_2, L=2, final_average_call=jnp.mean, tapering_function=tukey_taper)
axs[1][0].plot(x_ul, y_ul, color="black", alpha=1)
axs[1][0].plot(x_ll, y_ll, label="GW190521_074359, Tukey-windowed", color=red, alpha=0.5)
axs[1][0].legend(loc="lower left")


# Fill: Lower right plot
x_lr, y_lr, _ = calculate_welch_average(x=time_event_1, y=strain_event_1, L=1, final_average_call=jnp.mean, tapering_function=tukey_taper)
axs[1][1].plot(x_ul, y_ul, color="black", alpha=1)
axs[1][1].plot(x_lr, y_lr, label=r"GW150914, window size $1\:\mathrm{s}$", color=green, alpha=0.5)
axs[1][1].legend(loc="lower left")


# Label subplots to refer back to in Tex Doc

labels = ["(a)", "(b)", "(c)", "(d)"]

for ax, lab in zip(axs.flat, labels):
    ax.text(
        0.05, 0.95, lab,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=label_fontsize_pts,
    )


plt.tight_layout()
save_figure(save_fig=False)


## Tukey vs. Hann window

In [ ]:
import jax.numpy as jnp
from scipy.signal.windows import hann
from scipy.signal.windows import tukey

tukey_taper = lambda d: tukey(M=len(d), alpha=0.1, sym=True)
hann_taper = lambda d: hann(M=len(d), sym=True)


x = np.linspace(-1, 1, 500)
y = np.ones(500)

y_tukey = tukey_taper(y) * y
y_hann = hann_taper(y) * y

_ = plt.figure(figsize=(8,4))
plt.plot(x, y_tukey, label="Tukey window", color="black")
plt.plot(x, y_hann, label="Hann window", color=blue,)
plt.legend(loc="best")
thesis_plot(xl="$x$", yl="$y$", save_fig=False, mode="longer")



## Introduction to the Wigner function

In [ ]:
# Produce the data
n_pix = 2000
time = np.linspace(0, 1, n_pix)
dt = time[1] - time[0]

dirac_delta_time_norm = 1/np.sqrt(dt)

# Upper left
Xi_ul = np.random.standard_normal((n_pix,n_pix))

# Upper right
xi = np.random.standard_normal(n_pix) * dirac_delta_time_norm
S, t, f = Stress_jft(xi=xi, time=time, supress_print=True)

# Properly norm upper left
df = f[1]-f[0]
dirac_delta_freq_norm = 1/np.sqrt(df)
Xi_ul *= dirac_delta_freq_norm * dirac_delta_time_norm

In [ ]:
# Calculate figure width height dynamically based on desired padding between subplots such that subplots retain equal aspect ratio when
# padded through fig.subplot_adjust

nrows, ncols = 2, 2
w_sub, h_sub = 4, 4       # desired subplot size in inches
wspace, hspace = 0.3, 0.2 # fraction of subplot size for spacing

fig_width = ncols * w_sub + (ncols-1) * w_sub * wspace
fig_height = nrows * h_sub + (nrows-1) * h_sub * hspace

print("Figure width and height in inches: ", fig_width, fig_height)

In [ ]:
fig, axs = plt.subplots(ncols, nrows, figsize=(fig_width, fig_height), sharex=True, sharey=True, )
flattened_axs = axs.flatten()

for ax in axs.flatten():
    # If this comes after everything else, all subplots collapse for some reason
    ax.set_aspect('equal')  # forces 1:1 ratio per subplot
    # ax.legend(loc="best")

# Fill: Upper left plot
x_ul = Xi_ul/1e3
cb_1, im_1 = visualize_stress(stress_matrix=x_ul, rows=f, cols=t, smooth=False, custom_ax=flattened_axs[0], delay_plot=True, colorbar_label="", return_aux=True)
flattened_axs[0].set_title(r"White noise, unsmoothed")


# Fill: Upper right plot
x_ur = S.real/1e3
cb_2, im_2 = visualize_stress(stress_matrix=x_ur, rows=f, cols=t, smooth=False, custom_ax=flattened_axs[1], delay_plot=True, colorbar_label=r"Stress $\mathrm{[10^{-3}]}$", return_aux=True)
flattened_axs[1].set_title(r"White noise Wigner, unsmoothed")


# Fill: Lower left plot
x_ll = Xi_ul
cb_3, im_3 = visualize_stress(stress_matrix=x_ll, rows=f, cols=t, smooth=True, custom_ax=flattened_axs[2], delay_plot=True, colorbar_label="", return_aux=True)
flattened_axs[2].set_title(r"White noise, smoothed")


# Fill: Lower right plot
x_lr = smooth_matrix(S, smoothing_lvl=5).real
cb_4, im_4 = visualize_stress(stress_matrix=x_lr, rows=f, cols=t, smooth=False, custom_ax=flattened_axs[3], delay_plot=True, return_aux=True)
flattened_axs[3].set_title("White noise Wigner, smoothed")

# Make a bit more space for the colorbars
fig.subplots_adjust(wspace=wspace, hspace=hspace, right=0.88)  # right: 1 => Axes extend until 100% of the widtH

# Adjust upper row colorbar limits: fix them to wigner limits
min_data_upper_rows = np.min(x_ur)
max_data_upper_rows = np.max(x_ur)

im_1.set_clim(min_data_upper_rows, max_data_upper_rows)
cb_1.update_normal(im_1)

# Adjust lower row colorbar limits: fix them to wigner limits (tested: cmap as well as values do change)
min_data_lower_rows = np.min(x_lr)
max_data_lower_rows = np.max(x_lr)
im_3.set_clim(min_data_lower_rows, max_data_lower_rows)
cb_3.update_normal(im_3)

flattened_axs[0].set_ylabel(r"Frequency $\mathrm{[Hz]}$")
flattened_axs[2].set_ylabel(r"Frequency $\mathrm{[Hz]}$")
flattened_axs[2].set_xlabel(r"Time $\mathrm{[s]}$")
flattened_axs[3].set_xlabel(r"Time $\mathrm{[s]}$")

# Finally append labels for back reference

labels = ["(a)", "(b)", "(c)", "(d)"]

for ax, lab in zip(flattened_axs, labels):
    ax.text(
        0.05, 0.95, lab,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=label_fontsize_pts,
        bbox=dict(facecolor='white', edgecolor='white', alpha=0.6, lw=0)

    )

save_figure(False, tight_ly=False)

print("\n")
print("Mean of white noise Wigner: ", np.mean(S).real, " should be ~ 1")
print("Std of white noise Wigner: ", np.std(S).real, f" should be ~ {np.round(np.sqrt(n_pix),2)}")

## Wigner of numerical relativity template and whitened data

In [10]:
# Get the data

# Whitened data of GW150914
GW150914 = get_time_and_strain_from_disc(add_whitened_data=True)
time_GW_event = GW150914.event_time
xi_GW_event = GW150914.event_strain_white
S_gw, t_gw, f_gw = Stress_jft(xi=xi_GW_event, time=time_GW_event, supress_print=True)

# Num rel data
nrt_strain_values = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_strain_values.txt") * 1e19
nrt_time_values = np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_time_values.txt")
xi_nrt = nrt_strain_values
xi_time = nrt_time_values - nrt_time_values[0] - 1.393

S_nrt, t_nrt, f_nrt = Stress_jft(xi=xi_nrt, time=xi_time, supress_print=True)


Start: Calculating welch average

Constructing 16 windows over which we average.



In [20]:
# Write helper function

def detection_statistic(stress_matrix, raw=False, plot=True, time_var=None, custom_ax=None, title=None, normalize=False, lb=""):
    """

    :param stress_matrix: 2D array, shape (n_f, n_t):   Output from stress_jft() function in standard DFT order.
    :param raw: bool,                                   If True, the input matrix is not manipulated in any way.
    :param normalize: bool,                             If True, DC-line is divided by its max and the average is printed.

    Picks out the interference pattern on the DC-line of a Wigner-derived phase-space distribution through the following
    operations:

        1. Shifts from standard DFT order to DC-centered ordered along columns
        2. Smooths through Gaussian convolution
        3. Takes the absolute square. We call the resulting matrix 'smoothed Wigner power' (power => positive)
        4. Extracts DC-line from smoothed Wigner power

    :return dc_line,    The line corresponding to f=0 in the smoothed Wigner power
    :return SWP,        Smoothed Wigner power matrix in standard DFT order.

    """

    X = np.fft.fftshift(stress_matrix, axes=0)  # shift rows, corresponding to frequencies, such that f=0 is "in the middle" of the matrix. stress_matrix MUST be in standard DFT order (f=0 at the very bottom/top)

    if raw:
        SWP = X
    else:
        SWP = smooth_matrix(X, smoothing_lvl=5).real**2

    where_dc = SWP.shape[0]//2
    dc_line = SWP[where_dc, :]

    if normalize:
        dc_line /= np.max(dc_line)
        print("Average of smoothed Wigner power's DC-line: ", np.average(dc_line))

    if plot:
        if time_var is None:
            raise ValueError("To plot, please provide time array")
        if custom_ax is None:
            _ = plt.figure()
            axis = plt.gca()
        else:
            axis = custom_ax
        axis.plot(time_var, dc_line, label=lb, color="black")
    return dc_line, np.fft.ifftshift(SWP, axes=0)

In [6]:
# Calculate figure width height dynamically based on desired padding between subplots such that subplots retain equal aspect ratio when
# padded through fig.subplot_adjust

nrows, ncols = 3, 2
w_sub, h_sub = 4, 4       # desired subplot size in inches
wspace, hspace = 0.3, 0.2 # fraction of subplot size for spacing

fig_width = ncols * w_sub + (ncols-1) * w_sub * wspace
fig_height = nrows * h_sub + (nrows-1) * h_sub * hspace

print("Figure width and height in inches: ", fig_width, fig_height)

Figure width and height in inches:  9.2 13.6


In [41]:
# Build the sceleton

fig, axs = plt.subplots(nrows, ncols, figsize=(fig_width, fig_height), sharex=False, sharey=False, )
flattened_axs = axs.flatten()

for ax in axs.flatten():
    # If this comes after everything else, all subplots collapse for some reason
    ax.set_aspect('equal')  # forces 1:1 ratio per subplot

# Fill: Upper left plot
x_ul = S_nrt.real/np.max(S_nrt.real)
cb_1, im_1 = visualize_stress(stress_matrix=x_ul, rows=f_nrt, cols=t_nrt, smooth=False, custom_ax=flattened_axs[0], delay_plot=True, colorbar_label="", return_aux=True)
# flattened_axs[0].set_title(r"$S_{ft}$ of NRT, unsmoothed")
flattened_axs[0].set_title(r"Numerical relativity")


# Fill: Upper right plot
x_ur = S_gw.real/np.max(S_gw.real)
cb_2, im_2 = visualize_stress(stress_matrix=x_ur, rows=f_gw, cols=t_gw, smooth=False, custom_ax=flattened_axs[1], delay_plot=True, colorbar_label=r"Stress (arb. units)", return_aux=True)
# flattened_axs[1].set_title(r"$S_{ft}$ of whitened GW150914 data, unsmoothed")
flattened_axs[1].set_title(r"Whitened data")


# Fill: Lower left plot
x_ll = smooth_matrix(S_gw.real,5)
x_ll /= np.max(x_ll)
cb_3, im_3 = visualize_stress(stress_matrix=x_ll, rows=f_gw, cols=t_gw, smooth=False, custom_ax=flattened_axs[2], delay_plot=True, colorbar_label="", return_aux=True)
# flattened_axs[2].set_title(r"$S_{ft}$ of whitened GW150914 data, smoothed")
flattened_axs[2].set_title(r"As (b) but smoothed")


# Fill: Lower right plot
x_lr = smooth_matrix(S_gw.real, 5)**2
x_lr /= np.max(x_lr)
cb_4, im_4 = visualize_stress(stress_matrix=x_lr, rows=f_gw, cols=t_gw, smooth=False, custom_ax=flattened_axs[3], delay_plot=True, return_aux=True, colorbar_label="Stress (arb. units)")
flattened_axs[3].set_title(r"Square of (c)")


# Fill plot under lower right plot (lower-lower right)
x_llr = x_ll
x_llr /= np.max(x_llr)
_ = detection_statistic(stress_matrix=x_llr, time_var=t_gw, custom_ax=flattened_axs[5])
flattened_axs[5].set_title(r"$f=0$ (DC) line of (d)")
flattened_axs[5].set_aspect('auto')  # otherwise collapses because set_aspect is globally set to 'equal' but this plot has different axes ranges

# Get rid of unneccessary axis
flattened_axs[4].set_axis_off()

# Make a bit more space for the colorbars
fig.subplots_adjust(wspace=wspace, hspace=hspace, right=0.88)  # right: 1 => Axes extend until 100% of the width

# Adjust upper row colorbar limits: fix them to wigner limits
min_data = np.min(x_ul)
max_data = np.max(x_ul)

im_2.set_clim(min_data, max_data)
im_3.set_clim(min_data, max_data)
cb_2.update_normal(im_1)
cb_3.update_normal(im_1)


flattened_axs[0].set_ylabel(r"Frequency $\mathrm{[Hz]}$")
flattened_axs[2].set_ylabel(r"Frequency $\mathrm{[Hz]}$")
flattened_axs[5].set_ylabel(r"$(\mathcal{G}\ast S)^2\vert_{f=0}$")

# flattened_axs[3].set_xlabel(r"Time $\mathrm{[s]}$")
flattened_axs[2].set_xlabel(r"Time $\mathrm{[s]}$")
flattened_axs[5].set_xlabel(r"Time $\mathrm{[s]}$")

flattened_axs[1].set_yticklabels([])
flattened_axs[3].set_yticklabels([])

# Set x and y limits
independent_ax = flattened_axs[5]
for ax in flattened_axs:
    if ax is not independent_ax:
        ax.set_xlim(-.14,.1)
        ax.set_ylim(-350,350)
    else:
        ax.set_xlim(-.14,.1)

# Finally append labels for back reference

labels = ["(a)", "(b)", "(c)", "(d)", "", "(e)"]

for ax, lab in zip(flattened_axs, labels):
    ax.text(
        0.05, 0.95, lab,
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=label_fontsize_pts,
        bbox=dict(facecolor='white', edgecolor='white', alpha=0.6, lw=0)

    )

save_figure(True, tight_ly=False)


		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


## Detection statistic over full domain

In [16]:
# Get the data
from phase_II.nifty_re_playground.strain_tools import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib tk

# Whitened data of GW150914
GW150914_tmp = get_time_and_strain_from_disc(add_whitened_data=True, event_name="GW150914", detector="H1", data_duration="32sec")
time_GW_event_tmp = GW150914_tmp.event_time
xi_GW_event_tmp = GW150914_tmp.event_strain_white
S_gw_tmp, t_gw_tmp, f_gw_tmp = Stress_jft(xi=xi_GW_event_tmp, time=time_GW_event_tmp, supress_print=True)

# plt.plot(GW150914_tmp.event_time, GW150914_tmp.event_strain_white)
# plt.show()
visualize_stress(S_gw_tmp,f_gw_tmp, t_gw_tmp, smooth=True)


Start: Calculating welch average

Constructing 16 windows over which we average.

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [21]:
_ = detection_statistic(S_gw_tmp, time_var=t_gw_tmp)

## Detection statistic for smooth zeropading